In [2]:
# loading my cleaned data
import numpy as np
import torch

X_train = np.load('X_train.npy')
X_test = np.load('X_test.npy')
y_train = np.load('y_train.npy')
y_test = np.load('y_test.npy')

X_train.shape, X_test.shape

((168834, 53), (42209, 53))

In [4]:
# converting to tensors, same pattern as my toy example
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

X_train_t.shape, y_train_t.shape

(torch.Size([168834, 53]), torch.Size([168834, 1]))

In [6]:
# same model shape as my toy example, just bigger input
import torch.nn as nn

class IDSNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x

model = IDSNet(input_dim=X_train_t.shape[1])
print(model)

IDSNet(
  (fc1): Linear(in_features=53, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=16, bias=True)
  (fc3): Linear(in_features=16, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)


In [8]:
# batching the data so training is efficient
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

len(train_loader)

660

In [10]:
import torch.optim as optim

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [12]:
# training loop, same pattern as before, now with batches
for epoch in range(10):
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Avg Loss: {avg_loss:.4f}")

Epoch 1, Avg Loss: 0.1575
Epoch 2, Avg Loss: 0.0650
Epoch 3, Avg Loss: 0.0565
Epoch 4, Avg Loss: 0.0515
Epoch 5, Avg Loss: 0.0479
Epoch 6, Avg Loss: 0.0445
Epoch 7, Avg Loss: 0.0412
Epoch 8, Avg Loss: 0.0373
Epoch 9, Avg Loss: 0.0346
Epoch 10, Avg Loss: 0.0316


In [14]:
# evaluating on test set
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

model.eval()
with torch.no_grad():
    y_pred_probs = model(X_test_t)
    y_pred = (y_pred_probs > 0.5).float()

y_pred_np = y_pred.numpy()
y_test_np = y_test_t.numpy()

print("Accuracy:", accuracy_score(y_test_np, y_pred_np))
print("Precision:", precision_score(y_test_np, y_pred_np))
print("Recall:", recall_score(y_test_np, y_pred_np))
print("F1:", f1_score(y_test_np, y_pred_np))
print("Confusion matrix:\n", confusion_matrix(y_test_np, y_pred_np))

Accuracy: 0.9926082115188704
Precision: 0.992887184562098
Recall: 0.997452466757798
F1: 0.9951645899200298
Confusion matrix:
 [[ 9791   230]
 [   82 32106]]


In [16]:
#Deep learning baseline: 99.26% accuracy, 99.52% F1 — slightly below the Random Forest (99.89%/99.93%). This matches known behavior in ML literature: tree-based ensembles often outperform neural nets on structured tabular data. Deep learning remains the right choice going forward since it's what federated learning frameworks (Flower) are built around, and it's the architecture your proposal targets.

In [18]:
# saving the trained model
torch.save(model.state_dict(), 'centralized_model.pth')